# 07 — Baixar e preparar CIPLab e Human Faces
Execute de cima para baixo. **CPU é suficiente.** Este notebook baixa as duas bases oficiais do Kaggle, conserva os arquivos no Drive, verifica leitura, identifica classes pelas pastas e remove duplicatas exatas por pixels RGB. Não gera rostos nem infere raça/gênero.

As duas bases serão testes externos: não serão usadas para escolher limiar, época ou hiperparâmetros. Não são pares de pessoas correspondentes. CIPLab contém composições por Photoshop; a técnica de Human Faces deve ser confirmada no README da versão baixada. Os totais são calculados dos arquivos, não copiados do pôster antigo.

Se a sessão cair, execute novamente desde o começo: downloads concluídos e registros de leitura serão reaproveitados. Não execute simultaneamente duas cópias deste notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/tcc_justica')
BASE = ROOT/'dados_externos'
BASE.mkdir(parents=True, exist_ok=True)
print('Pasta persistente:', BASE)

In [ ]:
%pip -q install kagglehub pandas pillow tqdm matplotlib

## 1. Configuração e autenticação opcional
Tente primeiro com `LOGIN_KAGGLE=False`. Se o Kaggle exigir autenticação, aceite os termos na página da base, altere para `True` e execute novamente. O login solicita o token de forma interativa. Não cole tokens em células nem envie suas credenciais aqui.

In [ ]:
import os, json, hashlib, shutil, re, unicodedata, time
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import kagglehub
LOGIN_KAGGLE = False
if LOGIN_KAGGLE:
    kagglehub.login()
HANDLES = {
    'ciplab': 'ciplab/real-and-fake-face-detection',
    'human_faces': 'kaustubhdhote/human-faces-dataset',
}
EXTENSIONS = {'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff'}

## 2. Baixar e copiar para o Drive

In [ ]:
for name, handle in HANDLES.items():
    dest = BASE/name
    marker = BASE/f'{name}_download.json'
    if marker.exists() and dest.exists() and any(dest.rglob('*')):
        print(name, ': download já salvo.')
        continue
    print('Baixando:', handle)
    try:
        downloaded = Path(kagglehub.dataset_download(handle))
    except Exception as exc:
        raise RuntimeError('Download falhou. Confira a mensagem, os termos da base e a autenticação opcional acima; depois repita esta célula.') from exc
    shutil.copytree(downloaded, dest, dirs_exist_ok=True)
    assert any(p.suffix.lower() in EXTENSIONS for p in dest.rglob('*')), 'Nenhuma imagem encontrada; confira o conteúdo baixado.'
    marker.write_text(json.dumps({'handle':handle,'cache_source':str(downloaded),'saved_at':time.strftime('%Y-%m-%dT%H:%M:%S'),'kagglehub_version':kagglehub.__version__},indent=2))
    print('Salvo:', dest)

## 3. Conferir as pastas e a documentação
A tabela mostra os diretórios que contêm imagens. O mapeamento abaixo aceita nomes explícitos de classe; nomes desconhecidos nunca são adivinhados pelo rosto ou pelo nome do arquivo. Se a etapa seguinte interromper, veja `rotulos_nao_resolvidos.csv` e acrescente o nome exato da pasta a `CLASS_MAP`, após conferir o significado na documentação.

In [ ]:
for name in HANDLES:
    folder=BASE/name
    paths=sorted(p for p in folder.rglob('*') if p.suffix.lower() in EXTENSIONS)
    display(pd.Series([str(p.parent.relative_to(folder)) for p in paths]).value_counts().rename('imagens').to_frame())
    for p in sorted(folder.rglob('*')):
        if p.is_file() and ('readme' in p.name.lower() or 'license' in p.name.lower()) and p.suffix.lower() in {'.txt','.md',''}:
            print(name, p.name, '\n', p.read_text(errors='replace')[:8000])

In [ ]:
CLASS_MAP = {
    'training_real':0, 'real':0, 'real_faces':0, 'real_images':0,
    'training_fake':1, 'fake':1, 'fake_faces':1, 'fake_images':1,
    'ai_generated':1, 'ai_generated_faces':1, 'ai_generated_images':1,
}
def folder_key(s):
    s=unicodedata.normalize('NFKD',s).encode('ascii','ignore').decode().lower()
    return re.sub(r'[^a-z0-9]+','_',s).strip('_')
def get_label(relative_path):
    labels={CLASS_MAP[folder_key(part)] for part in Path(relative_path).parts[:-1] if folder_key(part) in CLASS_MAP}
    return next(iter(labels)) if len(labels)==1 else None

def rgb_hash(image):
    rgb=image.convert('RGB')
    return hashlib.sha256(str(rgb.size).encode()+rgb.tobytes()).hexdigest()

records=[]; unresolved=[]
cache_path=BASE/'verificacao_cache.jsonl'
cache={}
if cache_path.exists():
    for line in cache_path.read_text().splitlines():
        try:
            r=json.loads(line);cache[r['image_path']]=r
        except json.JSONDecodeError: pass
with cache_path.open('a') as cache_file:
    for name in HANDLES:
        folder=BASE/name
        for path in tqdm(sorted(p for p in folder.rglob('*') if p.suffix.lower() in EXTENSIONS),desc=name):
            rel=str(path.relative_to(folder));label=get_label(rel)
            if label is None:
                unresolved.append({'dataset':name,'relative_path':rel});continue
            stat=path.stat();r=cache.get(str(path))
            if r is None or r['bytes']!=stat.st_size or r['mtime_ns']!=stat.st_mtime_ns:
                r={'image_path':str(path),'bytes':stat.st_size,'mtime_ns':stat.st_mtime_ns}
                try:
                    with Image.open(path) as im:
                        im.load();r.update(sha256_pixels=rgb_hash(im),width=im.width,height=im.height,error='')
                except Exception as exc:r.update(error=str(exc))
                cache_file.write(json.dumps(r)+'\n');cache_file.flush()
            r=r.copy();r.update(dataset=name,label=label,relative_path=rel,
                image_id=hashlib.sha256((name+'/'+rel).encode()).hexdigest(),
                race=None,gender=None,annotation_source='not_available',split='external_test')
            records.append(r)
pd.DataFrame(unresolved,columns=['dataset','relative_path']).to_csv(BASE/'rotulos_nao_resolvidos.csv',index=False)
assert not unresolved, f'{len(unresolved)} arquivos sem classe inequívoca. Ajuste CLASS_MAP; nada foi rotulado automaticamente.'
raw=pd.DataFrame(records)
assert len(raw), 'Nenhuma imagem rotulada.'
raw.to_csv(BASE/'inventario_completo.csv',index=False)
raw[raw.error!=''].to_csv(BASE/'imagens_invalidas.csv',index=False)
good=raw[raw.error==''].copy()
conflicts=good.groupby('sha256_pixels').label.nunique()
bad_hash=set(conflicts[conflicts>1].index)
good[good.sha256_pixels.isin(bad_hash)].to_csv(BASE/'conflitos_rotulo.csv',index=False)
good=good[~good.sha256_pixels.isin(bad_hash)].copy()
# Mantém uma ocorrência por base; duplicatas entre bases são registradas, não contadas como independentes.
good[good.duplicated('sha256_pixels',keep=False)].to_csv(BASE/'duplicatas.csv',index=False)
good=good.drop_duplicates(['dataset','sha256_pixels']).sort_values(['dataset','image_id'])
for name in HANDLES:
    assert set(good.loc[good.dataset==name,'label'])=={0,1}, f'{name}: confira as duas classes.'
good.to_csv(BASE/'manifesto_externo.csv',index=False)
summary=good.groupby(['dataset','label']).size().rename('n').reset_index()
summary.to_csv(BASE/'contagens.csv',index=False)
display(summary)
print('Pronto:',BASE/'manifesto_externo.csv')

## 4. Inspeção visual e resumo para enviar

In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(4,4,figsize=(10,10),squeeze=False)
for row,(key,group) in enumerate(good.groupby(['dataset','label'])):
    sample=group.sample(min(4,len(group)),random_state=42)
    for col,ax in enumerate(axes[row]):
        ax.axis('off')
        if col<len(sample):
            r=sample.iloc[col]
            with Image.open(r.image_path) as im:ax.imshow(im.convert('RGB'))
            ax.set_title(f'{key[0]} / '+('real' if key[1]==0 else 'falsa'),fontsize=9)
plt.tight_layout();plt.savefig(BASE/'inspecao_externa.png',dpi=160);plt.show()
print('Envie: contagens.csv e inspecao_externa.png. Depois abra o notebook 08.')

### Fontes e limites
- https://www.kaggle.com/datasets/ciplab/real-and-fake-face-detection
- https://github.com/minostauros/Real-and-Fake-Face-Detection
- https://www.kaggle.com/datasets/kaustubhdhote/human-faces-dataset
- Download oficial: https://github.com/Kaggle/kagglehub

A remoção por hash identifica cópias exatas em RGB, não garante pessoas distintas nem elimina imagens quase iguais. Nenhum rótulo demográfico foi inferido. Preserve a versão, documentação e condições de uso das bases. O painel não é uma avaliação de predições.